# Generating-Unit Outage Screen with Logistic Regression – Solution

**Short name (GitHub):** `Energy_LogReg`  
Worked key. Numbers from scikit-learn on `data/energy_units.csv` (12,000 rows), `random_state=1`, `test_size=0.2`.

**Not an operating tool.** Synthetic unit-days. Yes = forced outage *or material derate*.

| Metric | Value |
|--------|-------|
| Class mix | 74.07% No / 25.93% Yes |
| X after dummies | 12,000 × 16 |
| Intercept | ≈ −2.08 |
| Test accuracy | ≈ 0.779 (No baseline on test ≈ 0.751) |
| Test confusion | TN 1739, FP 64, FN 467, TP 130 |
| Event precision / recall / F1 | 0.67 / 0.22 / 0.33 |
| ROC AUC | ≈ 0.757 |
| L1 zeros | 4 of 16 |

## Inline cheat-sheet

Dropped fuel reference = **coal**. Recover: age +, peak +, days-since-maint +, hydro/solar/nuclear −. Expanded card: `maint_overdue` ≈ +0.78.

## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load

In [ ]:
df = pd.read_csv("data/energy_units.csv")
print(df.head())
print(df.shape)
print("unique units", df.unit_id.nunique())

## 2. Imbalance

Always-predict-No baseline ≈ 0.741. Lesson accuracy 0.779 is real lift — still hide it behind recall/AUC when FN is a dark plant at peak.

In [ ]:
print(df.forced_outage.value_counts())
print(df.forced_outage.value_counts(normalize=True).round(4))

## 2.2 Dummy-encode

In [ ]:
feature_cols = [
    "age_years", "ambient_c", "capacity_mw", "load_factor",
    "days_since_maint", "is_peak_hour", "fuel", "region",
]
X = pd.get_dummies(df[feature_cols], drop_first=True).astype(float)
print(X.shape)
print(list(X.columns))

## 2.3 Heatmap

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Feature correlation (dummy-encoded X)")
plt.tight_layout()
plt.show()

## 2.4 Scale + y

`capacity_mw` spans ~20–1,400; `age_years` 1–62; peak is 0/1. Unscaled L1 converges; scale when you brief per-SD effects.

In [ ]:
for c in ["age_years", "ambient_c", "capacity_mw", "load_factor", "days_since_maint"]:
    print(f"{c:18s} min={X[c].min():8.2f}  max={X[c].max():8.2f}  mean={X[c].mean():8.2f}")
y = np.where(df.forced_outage == "No", 0, 1)
print("event rate", round(y.mean(), 4))

## 3. Fit

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, random_state=1, test_size=0.2
)
log_reg = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
log_reg.fit(x_train, y_train)
y_pred = log_reg.predict(x_test)
print(x_train.shape, x_test.shape, round(y_train.mean(), 3), round(y_test.mean(), 3))

### Parameters

Intercept ≈ −2.08. Hydro / solar / nuclear log-odds well below coal. Peak hour ≈ +0.38. Age ≈ +0.045 per year.

In [ ]:
print("Model Parameters, Intercept:")
print(log_reg.intercept_[0])
print("Model Parameters, Coeff:")
print(log_reg.coef_)

### Confusion + accuracy

[[1739, 64], [467, 130]]. Accuracy ≈ 0.779. At t = 0.5 the model flags only 194 of 2,400 days — a tight reserve list that misses most events.

In [ ]:
print("Confusion Matrix on test set:")
print(confusion_matrix(y_test, y_pred))
print("Accuracy Score on test set:")
print(log_reg.score(x_test, y_test))
print(classification_report(y_test, y_pred, digits=3))

## 4. Coef table + bar

In [ ]:
coef_df = (
    pd.DataFrame({"var": x_train.columns, "coef": log_reg.coef_[0]})
    .query("coef.abs() > 0")
    .sort_values("coef")
)
print(coef_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=coef_df, x="var", y="coef", color="#2c7bb6")
plt.xticks(rotation=90)
plt.title("LR Coefficient Values (forced outage)")
plt.tight_layout()
plt.show()

## 5. ROC — AUC ≈ 0.757

In [ ]:
y_pred_prob = log_reg.predict_proba(x_test)
roc_auc = roc_auc_score(y_test, y_pred_prob[:, 1])
print("ROC AUC score:", roc_auc)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob[:, 1])
plt.figure()
plt.plot(fpr, tpr, color="darkorange", label="ROC curve (area = %0.2f)" % roc_auc)
plt.plot([0, 1], [0, 1], color="navy", linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — forced outage")
plt.grid(True, alpha=0.3)
plt.legend(loc="lower right")
plt.show()

## 6. Alternates

In [ ]:
pipe = Pipeline([
    ("sc", StandardScaler()),
    ("lr", LogisticRegression(C=0.05, penalty="l1", solver="liblinear")),
])
pipe.fit(x_train, y_train)
p = pipe.predict_proba(x_test)[:, 1]
print("scaled acc", pipe.score(x_test, y_test))
print("scaled auc", roc_auc_score(y_test, p))
print(pd.DataFrame({
    "var": x_train.columns,
    "scaled_coef": pipe.named_steps["lr"].coef_[0],
}).sort_values("scaled_coef").to_string(index=False))

In [ ]:
log_l2 = LogisticRegression(max_iter=2000)
log_l2.fit(x_train, y_train)
print("L2 n_zero", int((np.abs(log_l2.coef_[0]) < 1e-12).sum()))
print("L2 acc", log_l2.score(x_test, y_test))
print("L2 auc", roc_auc_score(y_test, log_l2.predict_proba(x_test)[:, 1]))

### Age-only card — fuel and peak still add ranking power

In [ ]:
Xa = df[["age_years"]].astype(float)
xa_tr, xa_te, ya_tr, ya_te = train_test_split(Xa, y, random_state=1, test_size=0.2)
lr_a = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
lr_a.fit(xa_tr, ya_tr)
print("age-only acc", lr_a.score(xa_te, ya_te))
print("age-only AUC", roc_auc_score(ya_te, lr_a.predict_proba(xa_te)[:, 1]))
print("full-card AUC was", roc_auc)

### Threshold as reserve policy

In [ ]:
def predict_at(proba, t=0.5):
    return (proba >= t).astype(int)

p1 = y_pred_prob[:, 1]
print(f"{'t':>6} {'prec':>8} {'rec':>8} {'FP':>6} {'FN':>6} {'acc':>8}")
for t in (0.20, 0.30, 0.40, 0.50):
    pred = predict_at(p1, t)
    cm = confusion_matrix(y_test, pred)
    print(f"{t:6.2f} {precision_score(y_test, pred, zero_division=0):8.3f} "
          f"{recall_score(y_test, pred, zero_division=0):8.3f} "
          f"{cm[0,1]:6d} {cm[1,0]:6d} {accuracy_score(y_test, pred):8.3f}")

## 7. More practice

### Maintenance + operator card — `maint_overdue` dominates new terms

In [ ]:
cols2 = feature_cols + ["maint_overdue", "humidity", "operator"]
X2 = pd.get_dummies(df[cols2], drop_first=True).astype(float)
x2_tr, x2_te, y2_tr, y2_te = train_test_split(X2, y, random_state=1, test_size=0.2)
lr2 = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
lr2.fit(x2_tr, y2_tr)
print(pd.DataFrame({"var": x2_tr.columns, "coef": lr2.coef_[0]})
      .reindex(pd.Series(np.abs(lr2.coef_[0]), index=x2_tr.columns)
               .sort_values(ascending=False).index)
      .head(8))
print("expanded AUC", roc_auc_score(y2_te, lr2.predict_proba(x2_te)[:, 1]))
print("expanded rec", recall_score(y2_te, lr2.predict(x2_te)))

In [ ]:
log_bal = LogisticRegression(
    C=0.05, penalty="l1", solver="liblinear", class_weight="balanced"
)
log_bal.fit(x_train, y_train)
pred_b = log_bal.predict(x_test)
print("balanced acc", accuracy_score(y_test, pred_b))
print("balanced recall", recall_score(y_test, pred_b))
print("balanced prec", precision_score(y_test, pred_b))
print("balanced auc", roc_auc_score(y_test, log_bal.predict_proba(x_test)[:, 1]))
print(confusion_matrix(y_test, pred_b))

### Fuel slice — coal is the dropped reference (no dummy)

In [ ]:
nuc = x_test["fuel_nuclear"] == 1
wnd = x_test["fuel_wind"] == 1
# approximate coal-reference rows: all fuel dummies 0
fuel_dummies = [c for c in x_test.columns if c.startswith("fuel_")]
coal = x_test[fuel_dummies].sum(axis=1) == 0
pred = y_pred
for name, mask in [("coal-ref", coal), ("nuclear", nuc), ("wind", wnd)]:
    if mask.any():
        print(name, "n", int(mask.sum()),
              "base", round(y_test[mask].mean(), 3),
              "recall", round(recall_score(y_test[mask], pred[mask], zero_division=0), 3))

In [ ]:
iso_reserve = (
    "Recall (and a lower threshold): missing a peak-hour outage is a reserve shortfall."
)
capex_memo = (
    "Precision: do not write a coal-vs-nuclear story from a synthetic dummy that "
    "mostly reflects the training prior."
)
board_kpi = (
    "Accuracy 0.779 next to the 0.74 No baseline, plus AUC 0.76. Never show accuracy alone."
)
print(iso_reserve); print(capex_memo); print(board_kpi)

## 8. Simulation

In [ ]:
# --- editable parameters ---
C = 0.05
N = 6000
N_REPS = 12
NOISE = 0.00
T = 0.50
SEED = 1
# ---------------------------
rng = np.random.default_rng(SEED)
rows = []
for r in range(N_REPS):
    idx = rng.integers(0, len(X), size=N)
    Xs = X.iloc[idx].reset_index(drop=True)
    ys = y[idx].copy()
    xtr, xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=SEED + r)
    if NOISE > 0:
        flip = rng.random(len(ytr)) < NOISE
        ytr = ytr.copy()
        ytr[flip] = 1 - ytr[flip]
    m = LogisticRegression(C=C, penalty="l1", solver="liblinear", max_iter=2000)
    m.fit(xtr, ytr)
    p = m.predict_proba(xte)[:, 1]
    pred = (p >= T).astype(int)
    rows.append({
        "acc": accuracy_score(yte, pred),
        "recall": recall_score(yte, pred, zero_division=0),
        "prec": precision_score(yte, pred, zero_division=0),
        "auc": roc_auc_score(yte, p),
        "n_zero": int((np.abs(m.coef_[0]) < 1e-12).sum()),
    })
sim = pd.DataFrame(rows)
print(sim.describe().round(3))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, col in zip(axes, ["acc", "recall", "auc"]):
    ax.hist(sim[col], bins=8, color="#4c78a8", edgecolor="white")
    ax.set_title(col)
plt.suptitle(f"Energy_LogReg simulation  C={C}  N={N}  noise={NOISE}  T={T}")
plt.tight_layout()
plt.show()

## 9. Audience rewrite

In [ ]:
expert = (
    "L1-logistic (C=0.05, liblinear) on 16 dummy/continuous unit-day columns "
    "yields test AUC 0.757 and accuracy 0.779 versus a 0.751 No baseline on the same fold. "
    "At t=0.5 event-recall is 0.22 (TP=130, FN=467). Signs: age +0.045/year, peak +0.38, "
    "hydro/solar/nuclear well below the coal reference. Unit_id repeats, so MLE SEs are "
    "optimistic. This is a ranking screen for reserve, not a physical failure model."
)
technician = (
    "Read data/energy_units.csv. Dummy-encode age_years, ambient_c, capacity_mw, "
    "load_factor, days_since_maint, is_peak_hour, fuel, region (drop_first, astype float). "
    "Fit LogisticRegression(C=0.05, penalty='l1', solver='liblinear') on an 80/20 seed=1 split. "
    "Save intercept, non-zero coef table, CM [[1739,64],[467,130]], AUC. "
    "Production variant: StandardScaler pipeline plus a configurable reserve threshold."
)
executive = (
    "A simple outage screen is a few points better than always assuming the unit stays up. "
    "It ranks days reasonably (AUC about 0.76) and is conservative at the default cutoff — "
    "it flags about one in five actual events. Age, overdue maintenance and peak hour are "
    "the levers. Use it to sort the fleet for review, not to retire a fuel type. Synthetic file."
)
nonspecialist = (
    "We asked whether a power plant had a serious problem that day. Most days it did not. "
    "The model looks at how old the plant is, whether it is a peak hour, and what kind of "
    "fuel it uses. It is better than a coin flip at ranking the risky days and very cautious "
    "about raising an alarm. It is not a forecast of your lights going out tonight."
)
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)

## 10. Good fit vs not a fit

In [ ]:
good_fit = [
    "1. Binary outage / derate / peak-breach labels with a unit-day grain.",
    "2. Desk wants a sparse scorecard (age, fuel, peak, maintenance) not a black box.",
    "3. Teaching why 78% accuracy is close to 'always say no event'.",
    "4. Threshold workshops: reserve / no-action as a cutoff.",
    "5. Fuel and maintenance ablation before a more physical model.",
    "6. Baseline before survival analysis or gradient boosting on GADS-like files.",
    "7. Resource-adequacy education for mixed audiences.",
    "8. Monte-Carlo on mis-coded events and sample size before locking C.",
    "9. Comparing coal-reference dummies to nuclear/hydro without over-claiming causality.",
    "10. Showing operators that overdue maintenance shows up in the log-odds.",
]
not_a_fit = [
    "1. Live dispatch or protection settings. AUC 0.76 is not a relay.",
    "2. Treating repeated unit-days as independent when computing p-values.",
    "3. Causal 'coal is unreliable' headlines from a synthetic dummy versus coal reference.",
    "4. Customer billing / disconnection decisions.",
    "5. Multi-state derate MW as if it were a binary label (use a hurdle or beta model).",
]
for row in good_fit: print(row)
print("--- not a fit ---")
for row in not_a_fit: print(row)

## 11. Done checklist

- [x] 12,000-row synthetic unit-day blotter
- [x] 74/26 imbalance vs No baseline
- [x] 16-column dummy matrix
- [x] L1 model, CM [[1739,64],[467,130]], acc ≈ 0.779, AUC ≈ 0.757
- [x] Sparse coefs: age +, peak +, hydro/nuclear/solar − vs coal
- [x] Alternates, maintenance card, balanced weights, fuel slice, simulation
- [x] Four-audience rewrite + good-fit list + not-an-ops-tool disclaimer